In [1]:
import pandas as pd
import json
import numpy as np
filepath = 'C:/Users/Playdata/Downloads/대학교 자료/erd_data2/rdb_schools.json'

with open(filepath) as f:
    school_data = json.loads(f.read())

school_df = pd.DataFrame(school_data)
school_df.isna().sum()
# req join 용 
school_name = school_df

school_df = school_df[['name', 'country', 'address']]
school_df = school_df.rename(columns = {'address': 'location'})

school_df.to_csv('school_df.csv',index=False)
school_df


,name,country,location
0,New York University,USA,"New York, NY"
1,University of Southern California,USA,"Los Angeles, California"
2,University of Illinois Urbana-Champaign,USA,"Urbana, IL 61801"
3,Columbia University,USA,"New York, NY"
4,"University of California, Los Angeles",USA,"Los Angeles, CA"
5,Boston University,USA,"Boston, MA"
6,"University of California, Berkeley",USA,"Berkeley, CA"
7,"University of California, San Diego",USA,"La Jolla, CA"
8,Purdue University,USA,"West Lafayette, IN"
9,The Pennsylvania State University,USA,"University Park, PA"


In [2]:
import pandas as pd
from datetime import datetime

filepath = 'C:/Users/Playdata/Downloads/대학교 자료/erd_data2/rdb_admissions.json'

with open(filepath) as f:
    admissions_data = json.loads(f.read())

admissions_df = pd.DataFrame(admissions_data)
admissions_df.loc[admissions_df['admission_id']=='UCBerkeley_ADM', 'tuition']=79881
admissions_df.loc[admissions_df['admission_id']=='PennState_ADM', 'tuition']=92000
admissions_df
# 79,881  / 92000
# # school_df.isna().sum()
admissions_df['name'] = school_df['name']
admissions_df = admissions_df[['name', 'regular_deadline', 'early_deadline', 'tuition']]
admissions_df


# 날짜 전처리
current_year = datetime.now().year

def convert_deadline(series):
    with_year = series.apply(
        lambda x: f"{x} {current_year}" if isinstance(x, str) else x
    )
    as_dt = pd.to_datetime(with_year, errors='coerce')  # format 제거
    return as_dt.apply(lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) else None)

admission_df = admissions_df.copy()
admission_df['regular_deadline'] = convert_deadline(admission_df['regular_deadline'])
admission_df['early_deadline'] = convert_deadline(admission_df['early_deadline'])

admission_df.to_csv('admission_df.csv',index=False)
admission_df

,name,regular_deadline,early_deadline,tuition
0,New York University,2026-01-01,2026-11-01,60000.0
1,University of Southern California,2026-01-10,2026-11-01,75384.0
2,University of Illinois Urbana-Champaign,2026-01-05,2026-11-01,58316.0
3,Columbia University,2026-01-01,2026-11-01,70170.0
4,"University of California, Los Angeles",2026-12-01,None,80739.0
5,Boston University,2026-01-05,2026-11-02,73024.0
6,"University of California, Berkeley",2026-11-30,None,79881.0
7,"University of California, San Diego",2026-12-01,None,21546.0
8,Purdue University,2026-01-01,2026-11-01,30000.0
9,The Pennsylvania State University,2026-12-01,2026-11-01,92000.0


In [3]:
filepath = 'C:/Users/Playdata/Downloads/대학교 자료/erd_data2/rdb_requirements.json'

with open(filepath) as f:
    requirements_data = json.loads(f.read())

requirements_df = pd.DataFrame(requirements_data)
requirements_df


,document_type,requirement_policy,metric_name,metric_value,notes,requirement_id,school_code
0,TOEFL,Conditionally Required,Competitive Score,100,"No strict minimum, but this is the competitive...",NYU_REQ_1,NYU
1,IELTS,Conditionally Required,Competitive Score,7.5,"No strict minimum, but this is the competitive...",NYU_REQ_2,NYU
2,Duolingo,Conditionally Required,Competitive Score,135,"No strict minimum, but this is the competitive...",NYU_REQ_3,NYU
3,PTE,Conditionally Required,Competitive Score,70,"No strict minimum, but this is the competitive...",NYU_REQ_4,NYU
4,Cambridge English,Conditionally Required,Competitive Score,191,"No strict minimum, but this is the competitive...",NYU_REQ_5,NYU
...,...,...,...,...,...,...,...
62,IELTS,Conditionally Required,Competitive Score,6.5,"No strict minimum, but this is the competitive...",PennState_REQ_2,PennState
63,Duolingo,Conditionally Required,Competitive Score,105,"No strict minimum, but this is the competitive...",PennState_REQ_3,PennState
64,PTE,Conditionally Required,Competitive Score,53,"No strict minimum, but this is the competitive...",PennState_REQ_4,PennState
65,Cambridge English,Conditionally Required,Competitive Score,169,"No strict minimum, but this is the competitive...",PennState_REQ_5,PennState


In [4]:
# 대학교 이름 매칭
req_df = pd.merge(school_name,requirements_df, on = 'school_code', how = 'inner')
requirements_df = req_df[['name', 'document_type', 'metric_name', 'requirement_policy','metric_value', 'notes']]
requirement_df = requirements_df.rename(columns = {
                                 'document_type': 'type',
                                 'metric_name': 'metric_type',
                                 'requirement_policy': 'require',
                                 'metric_value' : 'value'
                                 }) 

requirement_df['value'][41]=125
requirement_df['value'][39]=90
requirement_df.to_csv('requirement_df.csv',index=False)
requirement_df


C:\Users\Playdata\AppData\Local\Temp\ipykernel_47064\3487388839.py:11: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  requirement_df['value'][41]=125
C:\Users\Playdata\AppData\Local\Temp\ipykernel_47064\3487388839.py:12: FutureWarning: Chaine

,name,type,metric_type,require,value,notes
0,New York University,TOEFL,Competitive Score,Conditionally Required,100,"No strict minimum, but this is the competitive..."
1,New York University,IELTS,Competitive Score,Conditionally Required,7.5,"No strict minimum, but this is the competitive..."
2,New York University,Duolingo,Competitive Score,Conditionally Required,135,"No strict minimum, but this is the competitive..."
3,New York University,PTE,Competitive Score,Conditionally Required,70,"No strict minimum, but this is the competitive..."
4,New York University,Cambridge English,Competitive Score,Conditionally Required,191,"No strict minimum, but this is the competitive..."
...,...,...,...,...,...,...
62,The Pennsylvania State University,IELTS,Competitive Score,Conditionally Required,6.5,"No strict minimum, but this is the competitive..."
63,The Pennsylvania State University,Duolingo,Competitive Score,Conditionally Required,105,"No strict minimum, but this is the competitive..."
64,The Pennsylvania State University,PTE,Competitive Score,Conditionally Required,53,"No strict minimum, but this is the competitive..."
65,The Pennsylvania State University,Cambridge English,Competitive Score,Conditionally Required,169,"No strict minimum, but this is the competitive..."


In [9]:
faq_df = pd.read_json('C:/Users/Playdata/Downloads/대학교 자료/erd_data2/rdb_faqs.json')
faq_df = pd.merge(school_name,faq_df, on = 'school_code', how = 'inner')
faq_df = faq_df[['name', 'question', 'answer', 'category']]
faq_df.to_csv('faq_df.csv', index = False)
faq_df


,name,question,answer,category
0,New York University,What are the key deadlines for undergraduate a...,"For undergraduate admissions at NYU, the key d...",Admissions
1,New York University,Is NYU test-optional?,"Yes, NYU is test-optional through the 2026-202...",Admissions
2,New York University,Does NYU meet 100% of demonstrated financial n...,"Yes, NYU will meet 100% of demonstrated financ...",Financial Aid
3,New York University,What English proficiency exams does NYU accept...,Non-native speakers may need to submit scores ...,International Applicants
4,New York University,What is the benefit of Spring Admission at NYU?,Spring Admission allows students to start thei...,Spring Admissions
5,University of Southern California,What are the important application deadlines f...,USC has the following key deadlines: Early Act...,Dates and Deadlines
6,University of Southern California,What is included in the application checklist ...,The application checklist for USC includes: Th...,Admission Requirements
7,University of Southern California,Does USC accept home-schooled students differe...,"Yes, home-schooled students applying to USC mu...",Application Process
8,University of Southern California,Does USC offer need-based financial aid to int...,"No, USC does not provide need-based financial ...",Financial Aid
9,University of Southern California,What are the English proficiency requirements ...,International applicants whose native language...,International Students
